# Frozen features or partial fine-tuning for bean-leaf classification?

This notebook reproduces a controlled comparison of **DINOv2-Small with a frozen backbone** and **DINOv2-Small with its final two blocks and final normalization fine-tuned**. Both arms train the same three-class linear head on the original Makerere Beans splits.

The workflow downloads pinned sources, audits images, checks the actual runtime, benchmarks and checks training, trains using validation-only selection, locks the protocol, and finally evaluates the held-out test set. DINOv2 is an established 2023 vision transformer, not a claim to the newest model in 2026. See `README.md` and `results/README.md` for the scope and execution evidence of the submitted study.

**Execution status:** this portable notebook is supplied without fabricated outputs. Colab execution is not established by local GPU execution. Run every cell in order in your own runtime; the saved JSON/CSV files document what actually ran. A lower score or an inconclusive difference is a valid result.


## 1. Locate the full project

In Colab, choose **Runtime → Change runtime type → GPU**. GPU availability is not guaranteed. Upload the complete project ZIP when prompted if the notebook was opened by itself. A repository checkout or an extracted project works directly. No account token or gated model approval is needed for the pinned DINOv2 checkpoint.


In [ ]:
from pathlib import Path
import csv, hashlib, importlib.metadata, json, os, shutil, subprocess, sys, zipfile
from datetime import datetime, timezone

IN_COLAB = "google.colab" in sys.modules or bool(os.environ.get("COLAB_RELEASE_TAG"))
candidates = [Path.cwd(), Path.cwd() / "beans_vit_project",
              Path.cwd() / "outputs" / "beans_vit_project", Path("/content/beans_vit_project")]
PROJECT_ROOT = next((p.resolve() for p in candidates if (p / "src/experiment.py").is_file()), None)
if PROJECT_ROOT is None and IN_COLAB:
    from google.colab import files
    print("Upload the complete beans_vit_project ZIP, containing src/ and requirements.txt.")
    uploaded = files.upload()
    archives = [Path(name) for name in uploaded if name.lower().endswith(".zip")]
    if len(archives) != 1:
        raise RuntimeError("Upload exactly one project ZIP.")
    target = Path("/content/beans_vit_project_upload").resolve()
    target.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(archives[0]) as archive:
        for entry in archive.infolist():
            destination = (target / entry.filename).resolve()
            if not destination.is_relative_to(target):
                raise ValueError("Unsafe ZIP path")
        archive.extractall(target)
    matches = [p.parent.parent for p in target.rglob("src/experiment.py")]
    if len(matches) != 1:
        raise RuntimeError("Expected one project with src/experiment.py in the ZIP.")
    PROJECT_ROOT = matches[0]
if PROJECT_ROOT is None:
    raise RuntimeError("Open this notebook from the extracted project folder or use the Colab upload option.")
assert (PROJECT_ROOT / "requirements.txt").is_file()
os.chdir(PROJECT_ROOT)
print("Project:", PROJECT_ROOT)
print("Python:", sys.version)


## 2. Install the pinned environment

The reference environment uses Python 3.11. This cell installs the exact core versions in `requirements.txt`. Colab may already contain a different PyTorch stack, so Colab also receives matching `torchvision==0.21.0` and `torchaudio==2.6.0` companions for `torch==2.6.0`. The project itself uses PIL rather than torchvision augmentation.

**If this cell changes packages that were already imported, restart the session and run from the top.** Do not import torch, NumPy or Transformers before this cell. Installation and model/data download time are additional to measured training time.


In [ ]:
install_command = [sys.executable, "-m", "pip", "install", "-r", str(PROJECT_ROOT / "requirements.txt")]
if IN_COLAB:
    install_command += ["torchvision==0.21.0", "torchaudio==2.6.0"]
subprocess.run(install_command, check=True)
os.environ["USE_TF"] = "0"
os.environ["USE_FLAX"] = "0"
print("Installation completed. Restart now if scientific packages were already imported in this session.")


## 3. Check the runtime and declare the study settings

Keep seed 17 because it is the prespecified detailed-analysis run. Three paired seeds are planned; reduce to `[17]` only if the measured budget requires it, before full training and test access, and report that limitation. Reduce the batch size if the pilot runs out of GPU memory. Changing settings after a run begins requires a new run ID; do not overwrite completed evidence.

`ALLOW_CPU` defaults to `False`: this stops early when a GPU was not allocated. Setting it to `True` explicitly allows the same experiment on CPU, with no promise about completion time.


In [ ]:
import torch
import numpy as np
from IPython.display import display, Image

ALLOW_CPU = False
SEEDS = [17, 42, 2026]
BATCH_SIZE = 8
MAX_EPOCHS = 10
PATIENCE = 3
HEAD_LR = 1e-3
BACKBONE_LR = 1e-5
WEIGHT_DECAY = 0.01
MODEL_ID = "facebook/dinov2-small"
MODEL_REVISION = "ed25f3a31f01632728cabb09d1542f84ab7b0056"
DATASET_REVISION = "27aa014ce09b193e1a6f58112d4a66e0eddb69c5"
# Retain this value when rerunning cells in the same kernel.
RUN_ID = globals().get("RUN_ID", "reproduction_" + datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ"))

if torch.__version__.split("+")[0] != "2.6.0" or np.__version__ != "2.2.6":
    raise RuntimeError("Imported versions differ from installed pins. Restart the runtime and run from the top.")
assert 17 in SEEDS and len(SEEDS) == len(set(SEEDS))
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu" and not ALLOW_CPU:
    raise RuntimeError("No CUDA GPU allocated. Select a GPU runtime, or explicitly set ALLOW_CPU=True.")
print("Device:", DEVICE, "| torch:", torch.__version__, "| CUDA build:", torch.version.cuda)
if DEVICE == "cuda":
    gpu = torch.cuda.get_device_properties(0)
    print("GPU:", gpu.name, "| VRAM GiB:", round(gpu.total_memory / 2**30, 2))

DATA_ROOT = PROJECT_ROOT / "work/data/beans"
IMAGE_ROOT = DATA_ROOT / "images"
STUDY = PROJECT_ROOT / "results/reproduction" / RUN_ID
AUDIT = STUDY / "data_audit"
RUNS = STUDY / "runs"
ANALYSIS = STUDY / "analysis"
LOCK = STUDY / "protocol.lock.json"
STUDY.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = str(PROJECT_ROOT / "work/huggingface")

def run(script, *args):
    command = [sys.executable, str(PROJECT_ROOT / "src" / script), *map(str, args)]
    print("Running", script, str(args[0]) if args else "")
    subprocess.run(command, cwd=PROJECT_ROOT, check=True)

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

def study_settings():
    return {"model_id": MODEL_ID, "model_revision": MODEL_REVISION,
            "dataset_revision": DATASET_REVISION, "seeds": SEEDS,
            "batch_size": BATCH_SIZE, "epochs": MAX_EPOCHS, "patience": PATIENCE,
            "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR, "weight_decay": WEIGHT_DECAY,
            "arms": ["frozen", "partial"], "reference_seed": 17, "primary_metric": "macro_f1",
            "bootstrap_resamples": 2000, "bootstrap_seed": 20261020,
            "sensitivity_checks": {"seed": 17, "arms": ["frozen", "partial"],
                                   "conditions": [{"name": "blur", "gaussian_radius": 1.0},
                                                  {"name": "brightness", "factor": 0.7}],
                                   "applied": "after 224x224 resizing, before normalization"}}
print("Fresh study directory:", STUDY)
print(json.dumps(study_settings(), indent=2))


## 4. Acquire and audit the original published splits

The script pins the dataset revision, verifies original ZIP hashes, decodes images, counts classes, computes file/pixel hashes and screens pHash candidates. It preserves the original 1,034/133/128 train/validation/test split. Test images may be inspected here for data integrity only, before any model predictions.

The bundled AI-assisted visual review judged 14 candidate pairs distinct. Its recorded decisions are reused only for matching image ID pairs in this pinned release. Unknown plant/farm/session dependence remains a limitation. New unresolved candidates require inspection before training; pHash alone does not establish duplication.


In [ ]:
if LOCK.exists() or any(RUNS.glob("*/seed_*/config.json")):
    raise RuntimeError("Do not rewrite the audit manifest after training starts; use a fresh run ID.")
AUDIT.mkdir(parents=True, exist_ok=True)
for name in ["review_decisions.csv", "visual_review_record.json"]:
    source = PROJECT_ROOT / "results/data_audit" / name
    if source.is_file():
        shutil.copy2(source, AUDIT / name)
run("prepare_data.py", "--data-root", DATA_ROOT, "--output-dir", AUDIT,
    "--review-decisions", AUDIT / "review_decisions.csv")
audit = json.loads((AUDIT / "audit_summary.json").read_text())
print(json.dumps({key: audit[key] for key in ["included_counts", "corrupt_images",
      "near_duplicate_candidate_pairs", "visual_review_status", "excluded_images"]}, indent=2))
unresolved = {key: count for key, count in audit["visual_review_status"].items()
              if key not in {"distinct", "confirmed_duplicate"} and count}
if unresolved:
    raise RuntimeError(f"Review unresolved image-pair decisions before proceeding: {unresolved}")
assert audit["dataset_revision"] == DATASET_REVISION
assert audit["included_counts"] == {"train": 1034, "validation": 133, "test": 128}
MANIFEST = AUDIT / "manifest.csv"
display(Image(filename=str(AUDIT / "dataset_examples.png"), width=850))


## 5. Verify gradients, checkpoint round trip and measured runtime

This pilot runs one complete training epoch for each arm. It checks that frozen parameters remain unchanged, the classifier learns, the selected backbone parameters update in partial fine-tuning, and saving/loading preserves logits. Pilot weights are discarded; full experiments restart from the pinned pretrained weights. No test prediction is produced.

Use the printed timing and GPU peak memory to judge the budget. The simple full-budget estimate below counts **training only** and omits validation, downloads, initialization, checkpointing and analysis. It is not a completion-time guarantee. If the pilot fails, resolve that issue before full training.


In [ ]:
if LOCK.exists():
    raise RuntimeError("The protocol is already locked; do not restart development in this study.")
common = ["--manifest", MANIFEST, "--data-root", IMAGE_ROOT, "--model-id", MODEL_ID,
          "--revision", MODEL_REVISION, "--device", DEVICE, "--batch-size", BATCH_SIZE,
          "--head-lr", HEAD_LR, "--backbone-lr", BACKBONE_LR, "--weight-decay", WEIGHT_DECAY]
run("experiment.py", "pilot", *common, "--output-dir", STUDY, "--seed", 17, "--benchmark-epoch")
pilot = json.loads((STUDY / "pilot/pilot_report.json").read_text())
assert {item["arm"] for item in pilot} == {"frozen", "partial"}
assert all(item["status"] == "passed" and item["epoch_benchmarked"] for item in pilot)
for item in pilot:
    print(item["arm"], "training seconds:", round(item["training"]["seconds"], 1),
          "peak allocated GPU GiB:", None if item["peak_cuda_allocated_bytes"] is None
          else round(item["peak_cuda_allocated_bytes"] / 2**30, 3))
estimate_minutes = sum(item["training"]["seconds"] for item in pilot) * MAX_EPOCHS * len(SEEDS) / 60
print("Training-only estimate if every run uses all epochs (minutes):", round(estimate_minutes, 1))


## 6. Train both arms; select checkpoints using validation only

Both arms use matched per-image augmentation, sample order and head initialization within each seed. Images are resized to 224×224 with bicubic interpolation and normalized using checkpoint settings. Training uses horizontal and vertical flips and rotations within ±15°. Validation and test use deterministic preparation.

Checkpoint selection prioritizes validation macro-F1, then validation loss, then earlier epoch. Only strict macro-F1 improvement resets patience. Frozen features means only the linear head learns; partial fine-tuning updates the last two transformer blocks and final normalization as well. It is not full-backbone fine-tuning or zero-shot prediction.


In [ ]:
if LOCK.exists():
    raise RuntimeError("This study is locked; further training requires a new study directory.")
if any(RUNS.glob("*/seed_*/config.json")):
    raise RuntimeError("Training evidence already exists. Do not overwrite it; inspect it or start a fresh run ID.")
assert all(item["status"] == "passed" for item in pilot)
(STUDY / "planned_settings.json").write_text(json.dumps(study_settings(), indent=2) + "\n")
# Rebuild arguments so an intentional pre-training setting change is reflected.
common = ["--manifest", MANIFEST, "--data-root", IMAGE_ROOT, "--model-id", MODEL_ID,
          "--revision", MODEL_REVISION, "--device", DEVICE, "--batch-size", BATCH_SIZE,
          "--head-lr", HEAD_LR, "--backbone-lr", BACKBONE_LR, "--weight-decay", WEIGHT_DECAY]
run("experiment.py", "train", *common, "--output-dir", RUNS, "--arm", "both",
    "--seeds", ",".join(map(str, SEEDS)), "--epochs", MAX_EPOCHS, "--patience", PATIENCE)


In [ ]:
import pandas as pd
validation_summaries = []
for arm in ["frozen", "partial"]:
    for seed in SEEDS:
        folder = RUNS / arm / f"seed_{seed}"
        assert (folder / "summary.json").is_file() and (folder / "best.pt").is_file()
        summary = json.loads((folder / "summary.json").read_text())
        validation_summaries.append({key: summary[key] for key in ["arm", "seed", "best_epoch",
                                    "best_validation_macro_f1", "epochs_completed", "training_seconds"]})
display(pd.DataFrame(validation_summaries))
print("These are validation metrics. Test predictions have not been requested.")


### Prespecified synthetic sensitivity conditions

Before locking, inspect fixed validation examples under Gaussian blur (radius 1 pixel) and brightness ×0.7. These conditions are applied after resizing to 224×224 and before normalization, using the same code path for both arms. They are fixed in the settings above; they are not chosen using test scores. Final test checks use seed 17 only and are paired transformations of the same images, not new independent observations or field validation.


In [ ]:
from PIL import Image as PILImage, ImageFilter, ImageEnhance, ImageOps
if LOCK.exists():
    raise RuntimeError("Validation sensitivity review belongs before the protocol lock.")
with MANIFEST.open(newline="", encoding="utf-8") as file:
    manifest_rows = list(csv.DictReader(file))
validation_examples = [sorted([row for row in manifest_rows if row["split"] == "validation"
                      and int(row["label"]) == label and row["include"] in {"1", "true", "True"}],
                      key=lambda row: row["image_id"])[0] for label in range(3)]
preview = PILImage.new("RGB", (224 * 3, 224 * 3), "white")
for row_index, row in enumerate(validation_examples):
    with PILImage.open(IMAGE_ROOT / row["image_id"]) as source:
        original = ImageOps.exif_transpose(source).convert("RGB").resize((224, 224), PILImage.Resampling.BICUBIC)
    variants = [original, original.filter(ImageFilter.GaussianBlur(radius=1.0)),
                ImageEnhance.Brightness(original).enhance(0.7)]
    for column, variant in enumerate(variants):
        preview.paste(variant, (224 * column, 224 * row_index))
print("Columns: original | Gaussian blur radius 1 | brightness x0.7")
print("Rows: angular leaf spot | bean rust | healthy (validation images only)")
display(preview)
preview.save(STUDY / "validation_sensitivity_preview.png")


## 7. Lock the completed development protocol before test evaluation

This cell records the actual settings, selected checkpoints, source code and manifest checksums. It refuses to overwrite a lock or create one after test predictions exist. All planned paired runs must be complete. The lock documents chronology and provenance; it does not prevent a person from intentionally editing files.

After locking, treat test results as final evaluation. Do not return to model selection or augmentation tuning using those results. A separate new study requires an honest explanation of any previous test exposure.


In [ ]:
if LOCK.exists():
    raise RuntimeError("A lock already exists. Continue to evaluation without rewriting it.")
if any(RUNS.glob("*/seed_*/predictions_test*.csv")):
    raise RuntimeError("Test predictions already exist; cannot retrospectively claim a pre-test lock.")
planned = json.loads((STUDY / "planned_settings.json").read_text())
assert planned == study_settings(), "Settings changed after training; resolve the provenance before testing."
checkpoints = []
for arm in planned["arms"]:
    for seed in planned["seeds"]:
        folder = RUNS / arm / f"seed_{seed}"
        for name in ["summary.json", "config.json", "best.pt", "history.csv", "validation_predictions.csv"]:
            assert (folder / name).is_file(), f"Incomplete run: {folder / name}"
        config = json.loads((folder / "config.json").read_text())
        assert config["arm"] == arm and config["seed"] == seed
        assert config["model_id"] == MODEL_ID and config["requested_revision"] == MODEL_REVISION
        assert config["manifest_sha256"] == sha256(MANIFEST)
        assert config["training_code_sha256"] == sha256(PROJECT_ROOT / "src/experiment.py")
        for config_key, setting_key in [("batch_size", "batch_size"), ("max_epochs", "epochs"),
                                        ("patience", "patience"), ("head_lr", "head_lr"),
                                        ("backbone_lr", "backbone_lr"), ("weight_decay", "weight_decay")]:
            assert config[config_key] == planned[setting_key], f"Setting mismatch: {config_key}"
        checkpoints.append({"arm": arm, "seed": seed,
                            "path": str((folder / "best.pt").relative_to(STUDY)).replace("\\", "/"),
                            "sha256": sha256(folder / "best.pt"), "config_sha256": sha256(folder / "config.json"),
                            "summary_sha256": sha256(folder / "summary.json")})
lock = {"locked_utc": datetime.now(timezone.utc).isoformat(), "settings": planned,
        "manifest_sha256": sha256(MANIFEST), "selected_checkpoints": checkpoints,
        "source_sha256": {name: sha256(PROJECT_ROOT / "src" / name)
                          for name in ["prepare_data.py", "experiment.py", "analyse_results.py", "validate_protocol.py"]},
        "test_access": "No model test predictions in this reproduction before this lock",
        "data_integrity_review": "Pretraining image inspection used only for documented data audit",
        "validation_sensitivity_preview_sha256": sha256(STUDY / "validation_sensitivity_preview.png")}
with LOCK.open("x", encoding="utf-8") as file:
    json.dump(lock, file, indent=2)
    file.write("\n")
print("Protocol locked:", LOCK)
print("Lock SHA256:", sha256(LOCK))


## 8. Evaluate the held-out test set

Evaluate every selected checkpoint on the same included test images. The standalone validator verifies checkpoint membership, config, manifest and source hashes against the lock immediately before each evaluation. The evaluation script additionally verifies the training-manifest checksum and records checkpoint and protocol-lock hashes. Scores are written independently of training summaries. Softmax confidence is not a calibrated probability.


In [ ]:
lock = json.loads(LOCK.read_text())
sys.path.insert(0, str(PROJECT_ROOT / "src"))
from validate_protocol import assert_locked
assert sha256(MANIFEST) == lock["manifest_sha256"]
for name, expected_hash in lock["source_sha256"].items():
    assert sha256(PROJECT_ROOT / "src" / name) == expected_hash, f"Source changed after lock: {name}"
for selected in lock["selected_checkpoints"]:
    checkpoint = STUDY / selected["path"]
    assert sha256(checkpoint) == selected["sha256"]
    assert sha256(checkpoint.parent / "config.json") == selected["config_sha256"]
    assert_locked(MANIFEST, checkpoint, LOCK, training_code=PROJECT_ROOT / "src/experiment.py")
    run("experiment.py", "evaluate", "--manifest", MANIFEST, "--data-root", IMAGE_ROOT,
        "--checkpoint", checkpoint, "--output-dir", checkpoint.parent, "--split", "test",
        "--confirm-test-locked", "--protocol-lock", LOCK, "--device", DEVICE,
        "--batch-size", lock["settings"]["batch_size"])
# Prespecified paired sensitivity checks for the reference seed, after all clean evaluations.
sensitivity = lock["settings"]["sensitivity_checks"]
for selected in lock["selected_checkpoints"]:
    if selected["seed"] != sensitivity["seed"]:
        continue
    checkpoint = STUDY / selected["path"]
    for condition in sensitivity["conditions"]:
        assert_locked(MANIFEST, checkpoint, LOCK, training_code=PROJECT_ROOT / "src/experiment.py")
        run("experiment.py", "evaluate", "--manifest", MANIFEST, "--data-root", IMAGE_ROOT,
            "--checkpoint", checkpoint, "--output-dir", checkpoint.parent, "--split", "test",
            "--corruption", condition["name"], "--confirm-test-locked", "--protocol-lock", LOCK,
            "--device", DEVICE, "--batch-size", lock["settings"]["batch_size"])


## 9. Recompute metrics, uncertainty and figures

Report macro-F1, accuracy, class metrics, confusion matrices, a training-majority-class baseline, actual errors and measured training cost. The prespecified seed-17 comparison uses 2,000 paired class-stratified bootstrap resamples with seed 20261020. This conditional test-sample interval is separate from variation across training seeds; unknown plant/farm dependence is not corrected. A single-seed reproduction cannot estimate training-seed variation.

Error examples follow a reproducible selection rule in the analysis script. Use the actual errors to discuss missed disease and false alarms. Limited disease categories and geography, nuisance backgrounds, unavailable collection-group identifiers and unknown foundation-model pretraining overlap constrain practical conclusions. This benchmark does not establish demographic fairness or safe field diagnosis.


In [ ]:
run("analyse_results.py", "--runs", RUNS, "--manifest", MANIFEST, "--data-root", IMAGE_ROOT, "--out", ANALYSIS)
display(pd.read_csv(ANALYSIS / "run_metrics.csv"))
display(pd.read_csv(ANALYSIS / "comparison.csv"))
analysis = json.loads((ANALYSIS / "analysis.json").read_text())
print(json.dumps(analysis, indent=2))
for name in ["learning_curves", "confusion_matrices", "performance_and_cost", "sample_predictions"]:
    display(Image(filename=str(ANALYSIS / "figures" / (name + ".png")), width=950))
display(pd.read_csv(ANALYSIS / "stress_metrics.csv"))
display(Image(filename=str(ANALYSIS / "figures/stress_sensitivity.png"), width=850))


## 10. Save the evidence before a temporary runtime ends

The export contains analysis, audit, configurations, histories and predictions, but excludes raw images, model weights, environments and caches. It includes an execution-environment listing. Save checkpoints separately if you need to rerun predictions without retraining. The lightweight bundle is suitable for repository evidence; it is not a complete backup of a temporary Colab runtime.


In [ ]:
environment = subprocess.run([sys.executable, "-m", "pip", "freeze"], check=True, capture_output=True, text=True).stdout
(STUDY / "environment.txt").write_text(environment)
export = PROJECT_ROOT / "work" / (RUN_ID + "_results.zip")
export.parent.mkdir(parents=True, exist_ok=True)
allowed = {".json", ".csv", ".png", ".pdf", ".md", ".txt"}
with zipfile.ZipFile(export, "w", compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(STUDY.rglob("*")):
        if path.is_file() and path.suffix.lower() in allowed:
            archive.write(path, arcname=str(path.relative_to(STUDY)))
print("Saved evidence bundle:", export, "| MiB:", round(export.stat().st_size / 2**20, 2))
if IN_COLAB:
    from google.colab import files
    files.download(str(export))


## References and attribution

- [Makerere AI Lab: original bean disease dataset](https://github.com/AI-Lab-Makerere/ibean) and [pinned Hugging Face distribution](https://huggingface.co/datasets/AI-Lab-Makerere/beans/tree/27aa014ce09b193e1a6f58112d4a66e0eddb69c5). The source README reports 1,296 images; the audited distributed splits contain 1,295. The original repository specifies MIT licensing.
- [Oquab et al. (2023), DINOv2](https://arxiv.org/abs/2304.07193) and [DINOv2-Small checkpoint](https://huggingface.co/facebook/dinov2-small/tree/ed25f3a31f01632728cabb09d1542f84ab7b0056); model license Apache-2.0.
- [Dosovitskiy et al., An Image is Worth 16×16 Words](https://arxiv.org/abs/2010.11929).
- [Google Colab FAQ](https://research.google.com/colaboratory/faq.html) for runtime and GPU availability limitations.

Disclose code and AI assistance according to course rules. The visual duplicate review is AI-assisted, not an independent agricultural label validation. No experiment guarantees a particular mark; conclusions must be supported by actual saved results and their limits.
